# 1. Import and Install Dependencies

In [ ]:
# Dependencies already installed - no pip install needed
print("Libraries ready!")


In [2]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp

# 2. Keypoints using MP Holistic

In [3]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_face_mesh = mp.solutions.face_mesh

In [4]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [5]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION)
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

In [6]:
def draw_styled_landmarks(image, results):
    # Draw face connections
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
                             mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                             mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1))
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2))
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2))
    # Draw right hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
                             

# 3. Extract Keypoint Values

In [7]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

In [163]:
# Test action ordering consistency
import os

def get_recorded_actions(data_path):
    """Return alphabetically sorted list of action folders"""
    return sorted([d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))])

# Check training actions
actions_training = ['good', 'hello', 'thanks']  # This is what label_map was built with
print(f"Training actions:    {actions_training}")

# Check inference actions
actions_inference = get_recorded_actions('MP_Data')
print(f"Inference actions:   {actions_inference}")

# Check if match
print(f"Match: {actions_training == list(actions_inference)}")
print(f"Mapping index -> action:")
for idx, a in enumerate(actions_inference):
    print(f"  {idx} → {a}")


Training actions:    ['good', 'hello', 'thanks']
Inference actions:   ['good', 'hello', 'thanks']
Match: True
Mapping index -> action:
  0 → good
  1 → hello
  2 → thanks


# 4. Setup Folders for Collection

In [9]:

import random

DATA_PATH = 'MP_Data'
no_sequences = 60
sequence_length = 30

def get_recorded_actions(data_path):
    if not os.path.exists(data_path):
        return np.array([])
    actions = sorted([
        d for d in os.listdir(data_path)
        if os.path.isdir(os.path.join(data_path, d))
    ])
    return np.array(actions)

actions = get_recorded_actions(DATA_PATH)
print(f'✅ Found {len(actions)} signs: {list(actions)}')
print(f'no_sequences: {no_sequences} | sequence_length: {sequence_length}')

✅ Found 8 signs: ['good', 'hello', 'human', 'morning', 'rights', 'sign', 'thanks', 'we']
no_sequences: 60 | sequence_length: 30


In [18]:
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)

for action in actions:
    for sequence in range(no_sequences):  # ✅ removed start_folder
        os.makedirs(os.path.join(DATA_PATH, action, str(sequence)), exist_ok=True)

print("Folders created successfully!")
print(os.listdir(DATA_PATH))

Folders created successfully!
['good']


In [19]:
for action in actions:
    path = os.path.join(DATA_PATH, action)
    seqs = sorted([int(s) for s in os.listdir(path) if os.path.isdir(os.path.join(path, s))])
    print(f"{action}: {len(seqs)} sequences → {seqs}")

good: 60 sequences → [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]


In [20]:
for action in actions:
    seqs = os.listdir(os.path.join(DATA_PATH, action))
    print(f"{action}: {len(seqs)} folders")

good: 60 folders


In [21]:
import os
print(os.listdir(DATA_PATH)) # This should show 'hello', 'thanks', etc.

['good']


 5. Collect Keypoint Values for Training and Testing

In [160]:
DATA_PATH = 'MP_Data'
no_sequences = 50
sequence_length = 30
actions = get_recorded_actions(DATA_PATH)
print(f"Actions: {list(actions)}")

Actions: ['good', 'hello', 'thanks']


In [ ]:
import time
import os

# CHANGE THIS to whichever sign you want to record
current_action = 'good'  # ← change this each time
add_more_sequences = 50  # How many MORE sequences to record (non-overwriting)

# Find existing sequences for this action
action_path = os.path.join(DATA_PATH, current_action)
os.makedirs(action_path, exist_ok=True)

existing_sequences = []
if os.path.exists(action_path):
    existing_sequences = sorted([int(d) for d in os.listdir(action_path) 
                                 if os.path.isdir(os.path.join(action_path, d))])

start_seq = max(existing_sequences) + 1 if existing_sequences else 0
end_seq = start_seq + add_more_sequences

print(f"📹 Recording: {current_action}")
print(f"📊 Existing sequences: {len(existing_sequences)} (highest: {max(existing_sequences) if existing_sequences else 'none'})")
print(f"➕ Adding sequences: {start_seq} to {end_seq - 1} ({add_more_sequences} new)")
print(f"🚀 Get ready! Starting in 3 seconds...")
time.sleep(3)

# Create folders for NEW sequences only
for sequence in range(start_seq, end_seq):
    os.makedirs(os.path.join(DATA_PATH, current_action, str(sequence)), exist_ok=True)

cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for sequence in range(start_seq, end_seq):
        for frame_num in range(sequence_length):
            ret, frame = cap.read()
            if not ret:
                print(f"❌ Camera error at sequence {sequence}")
                break
            
            image, results = mediapipe_detection(frame, holistic)
            draw_styled_landmarks(image, results)
            
            if frame_num == 0:
                cv2.putText(image, 'STARTING COLLECTION', (120,200),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 4, cv2.LINE_AA)
                cv2.putText(image, f'Recording {current_action} - Video {sequence} of {end_seq-1}', (15,12),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1, cv2.LINE_AA)
                cv2.imshow('OpenCV Feed', image)
                cv2.waitKey(2000)
            else:
                progress = f'{sequence - start_seq + 1}/{add_more_sequences}'
                cv2.putText(image, f'{current_action} [{progress}] - Video {sequence}', (15,12),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1, cv2.LINE_AA)
                cv2.imshow('OpenCV Feed', image)
            
            keypoints = extract_keypoints(results)
            npy_path = os.path.join(DATA_PATH, current_action, str(sequence), str(frame_num))
            np.save(npy_path, keypoints)
            
            if cv2.waitKey(10) & 0xFF == ord('q'):
                print("⏹️  Recording stopped by user")
                break
        else:
            print(f"✅ Recorded {current_action} sequence {sequence}")

print(f"🎉 Done! Recorded {add_more_sequences} new sequences for {current_action}")
print(f"📊 Total sequences now: {len(os.listdir(action_path))}")

cap.release()
cv2.destroyAllWindows()


Recording: good
Get ready! Starting in 3 seconds...
Done recording: good


In [34]:
cap.release()
cv2.destroyAllWindows()

In [35]:
import tensorflow as tf
print(tf.__version__)
print("Keras:", tf.keras.__version__)

2.17.0
Keras: 3.14.0


In [36]:
import numpy as np
import os

# Check if we can load just one file as a test
test_path = os.path.join(DATA_PATH, 'hello', '0', '0.npy')

if os.path.exists(test_path):
    test_data = np.load(test_path)
    print(f"Success! Data loaded. Shape: {test_data.shape}")
else:
    print("File not found. Check if DATA_PATH is correct or if files were actually saved.")

Success! Data loaded. Shape: (1662,)


# 6. Preprocess Data and Create Labels and Features

In [164]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from collections import Counter

np.random.seed(42)
random.seed(42)

label_map = {label:num for num, label in enumerate(actions)}

sequences, labels = [], []
for action in actions:
    action_path = os.path.join(DATA_PATH, action)
    available_sequences = sorted([int(s) for s in os.listdir(action_path) if os.path.isdir(os.path.join(action_path, s))])
    for sequence in available_sequences:
        window = []
        seq_path = os.path.join(DATA_PATH, action, str(sequence))
        all_frames_exist = all(os.path.exists(os.path.join(seq_path, f"{frame_num}.npy")) for frame_num in range(sequence_length))
        if all_frames_exist:
            for frame_num in range(sequence_length):
                res = np.load(os.path.join(seq_path, "{}.npy".format(frame_num)))
                window.append(res)
            sequences.append(window)
            labels.append(label_map[action])

X = np.array(sequences)

labels_array = np.array(labels)
print(f"✅ Class counts: {Counter(labels_array)}")

y = to_categorical(labels_array).astype(int)
print(f"✅ Actions: {list(actions)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=labels_array,
)
print(f"✅ Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")


✅ Class counts: Counter({0: 50, 1: 50, 2: 50})
✅ Actions: ['good', 'hello', 'thanks']
✅ Train samples: 127 | Test samples: 23


# 7. Build and Train LSTM Neural Network

In [151]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import TensorBoard

In [152]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

In [153]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard

# Build model
model = Sequential([
    Input(shape=(30, 1662)),
    LSTM(64,  return_sequences=True,  activation='relu'),
    Dropout(0.35),
    LSTM(128, return_sequences=True,  activation='relu'),
    Dropout(0.35),
    LSTM(64,  return_sequences=False, activation='relu'),
    Dropout(0.25),
    Dense(64, activation='relu'),
    Dropout(0.25),
    Dense(32, activation='relu'),
    Dense(len(actions), activation='softmax')
])

model.compile(
    optimizer='Adam',
    loss='categorical_crossentropy',
    metrics=['categorical_accuracy']
)
print(f'✅ Model for {len(actions)} signs!')

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=25,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'best_action.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    patience=10,
    factor=0.5,
    min_lr=1e-6,
    verbose=1
)

# Train
model.fit(
    X_train, y_train,
    epochs=300,
    batch_size=16,
    shuffle=True,
    validation_split=0.1,
    callbacks=[tb_callback, early_stop, checkpoint, reduce_lr],
    verbose=1
)

model.save('action.h5')
print('✅ Saved!')

✅ Model for 3 signs!
Epoch 1/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - categorical_accuracy: 0.3871 - loss: 1.3316
Epoch 1: val_loss improved from None to 5.84053, saving model to best_action.h5



Epoch 1: finished saving model to best_action.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 13s 271ms/step - categorical_accuracy: 0.3596 - loss: 1.2714 - val_categorical_accuracy: 0.1538 - val_loss: 5.8405 - learning_rate: 0.0010
Epoch 2/300
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - categorical_accuracy: 0.4433 - loss: 2.5788
Epoch 2: val_loss improved from 5.84053 to 1.14863, saving model to best_action.h5



Epoch 2: finished saving model to best_action.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - categorical_accuracy: 0.4123 - loss: 1.6459 - val_categorical_accuracy: 0.2308 - val_loss: 1.1486 - learning_rate: 0.0010
Epoch 3/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - categorical_accuracy: 0.3759 - loss: 1.2507
Epoch 3: val_loss improved from 1.14863 to 1.07572, saving model to best_action.h5



Epoch 3: finished saving model to best_action.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 85ms/step - categorical_accuracy: 0.4123 - loss: 1.1595 - val_categorical_accuracy: 0.6923 - val_loss: 1.0757 - learning_rate: 0.0010
Epoch 4/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - categorical_accuracy: 0.2907 - loss: 1.1101
Epoch 4: val_loss did not improve from 1.07572
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - categorical_accuracy: 0.3596 - loss: 1.1001 - val_categorical_accuracy: 0.3077 - val_loss: 1.0761 - learning_rate: 0.0010
Epoch 5/300
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - categorical_accuracy: 0.3904 - loss: 1.0650
Epoch 5: val_loss did not improve from 1.07572
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - categorical_accuracy: 0.4035 - loss: 1.0530 - val_categorical_accuracy: 0.3077 - val_loss: 1.0984 - learning_rate: 0.0010
Epoch 6/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - categorical_accuracy: 0.5107 - loss: 1.0081
Epoch 6: val_loss improved from 1.07572 to 1.02688, saving model to best_action.h5


Epoch 6: finished saving model to best_action.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - categorical_accuracy: 0.4649 - loss: 0.9829 - val_categorical_accuracy: 0.2308 - val_loss: 1.0269 - learning_rate: 0.0010
Epoch 7/300
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - categorical_accuracy: 0.4517 - loss: 1.0436
Epoch 7: val_loss did not improve from 1.02688
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - categorical_accuracy: 0.4474 - loss: 1.0669 - val_categorical_accuracy: 0.2308 - val_loss: 1.0878 - learning_rate: 0.0010
Epoch 8/300
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - categorical_accuracy: 0.3203 - loss: 1.1013
Epoch 8: val_loss did not improve from 1.02688
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - categorical_accuracy: 0.3421 - loss: 1.0976 - val_categorical_accuracy: 0.2308 - val_loss: 1.0843 - learning_rate: 0.0010
Epoch 9/300
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - categorical_accuracy: 0.3545 - loss: 1.0933
Epoch 9: val_loss did not improve from 1.02688
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/st

...


Epoch 42: finished saving model to best_action.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 91ms/step - categorical_accuracy: 0.7544 - loss: 0.4019 - val_categorical_accuracy: 0.8462 - val_loss: 0.4058 - learning_rate: 2.5000e-04
Epoch 43/300
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - categorical_accuracy: 0.7033 - loss: 0.4215
Epoch 43: val_loss improved from 0.40576 to 0.39660, saving model to best_action.h5



Epoch 43: finished saving model to best_action.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - categorical_accuracy: 0.7544 - loss: 0.3660 - val_categorical_accuracy: 0.9231 - val_loss: 0.3966 - learning_rate: 2.5000e-04
Epoch 44/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - categorical_accuracy: 0.8250 - loss: 0.3122
Epoch 44: val_loss improved from 0.39660 to 0.32383, saving model to best_action.h5



Epoch 44: finished saving model to best_action.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - categorical_accuracy: 0.8421 - loss: 0.3314 - val_categorical_accuracy: 0.9231 - val_loss: 0.3238 - learning_rate: 2.5000e-04
Epoch 45/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - categorical_accuracy: 0.8425 - loss: 0.3058
Epoch 45: val_loss improved from 0.32383 to 0.29884, saving model to best_action.h5



Epoch 45: finished saving model to best_action.h5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - categorical_accuracy: 0.8246 - loss: 0.3070 - val_categorical_accuracy: 0.9231 - val_loss: 0.2988 - learning_rate: 2.5000e-04
Epoch 46/300
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - categorical_accuracy: 0.9299 - loss: 0.3319
Epoch 46: val_loss did not improve from 0.29884
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - categorical_accuracy: 0.9561 - loss: 0.2924 - val_categorical_accuracy: 0.9231 - val_loss: 0.3341 - learning_rate: 2.5000e-04
Epoch 47/300
7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - categorical_accuracy: 0.9159 - loss: 0.4471
Epoch 47: val_loss did not improve from 0.29884
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - categorical_accuracy: 0.9211 - loss: 0.4790 - val_categorical_accuracy: 0.3846 - val_loss: 0.7667 - learning_rate: 2.5000e-04
Epoch 48/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - categorical_accuracy: 0.9247 - loss: 0.4347
Epoch 48: val_loss did not improve from 0.29884
8/8 ━━━━━━━━━━━━

✅ Saved!


In [135]:
# Check what's actually in your MP_Data
import os
for action in os.listdir('MP_Data'):
    count = len(os.listdir(os.path.join('MP_Data', action)))
    print(f'{action}: {count} sequences')

good: 60 sequences
hello: 51 sequences
thanks: 50 sequences


In [165]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Get predictions on test set
yhat_probs = model.predict(X_test, verbose=0)
yhat = np.argmax(yhat_probs, axis=1)
ytrue = np.argmax(y_test, axis=1)

# Confusion matrix
print("✅ Confusion Matrix:")
cm = confusion_matrix(ytrue, yhat)
print(cm)
print()
print("Predicted →")
print("     good  hello  thanks")
for i, row in enumerate(cm):
    print(f"{actions[i]:7} {row[0]:5} {row[1]:5} {row[2]:5}")

print("\n✅ Classification Report:")
print(classification_report(ytrue, yhat, target_names=actions))

print(f"✅ Test Accuracy: {np.mean(ytrue == yhat)*100:.1f}%")


✅ Confusion Matrix:
[[7 0 0]
 [0 8 0]
 [0 0 8]]

Predicted →
     good  hello  thanks
good        7     0     0
hello       0     8     0
thanks      0     0     8

✅ Classification Report:
              precision    recall  f1-score   support

        good       1.00      1.00      1.00         7
       hello       1.00      1.00      1.00         8
      thanks       1.00      1.00      1.00         8

    accuracy                           1.00        23
   macro avg       1.00      1.00      1.00        23
weighted avg       1.00      1.00      1.00        23

✅ Test Accuracy: 100.0%


In [166]:
# Ensure model saved
model.save('action.h5')
print('✅ Model saved to action.h5')

# Load it back to verify
from tensorflow.keras.models import load_model
loaded = load_model('action.h5')
print('✅ Model reloaded and verified')
print(f'Model summary:')
loaded.summary()


✅ Model saved to action.h5


✅ Model reloaded and verified
Model summary:


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_30 (LSTM)                  │ (None, 30, 64)         │       442,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_31 (LSTM)                  │ (None, 30, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_32 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 596,677 (2.28 MB)

 Trainable params: 596,675 (2.28 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [154]:
# See exactly what accuracy per class is
from sklearn.metrics import classification_report

yhat_probs = model.predict(X_test, verbose=0)
ytrue = np.argmax(y_test, axis=1)
yhat  = np.argmax(yhat_probs, axis=1)

print(classification_report(ytrue, yhat, 
      target_names=actions))

              precision    recall  f1-score   support

        good       1.00      1.00      1.00         7
       hello       1.00      1.00      1.00         8
      thanks       1.00      1.00      1.00         8

    accuracy                           1.00        23
   macro avg       1.00      1.00      1.00        23
weighted avg       1.00      1.00      1.00        23



# 8. Make Predictions

In [137]:
res = model.predict(X_test)
print('Predicted:', actions[np.argmax(res[0])])  # 0 = first test sample
print('Actual:   ', actions[np.argmax(y_test[0])])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Predicted: hello
Actual:    hello


In [138]:
actions[np.argmax(res[4])]

'good'

In [139]:
actions[np.argmax(y_test[4])]

'good'

# 9. Save Weights

In [155]:
model.save('action.h5')
print("model saved!")

model saved!


In [58]:
del model

In [156]:
from tensorflow.keras.models import load_model

model = load_model('action.h5')
print("✅ Model loaded successfully!")

✅ Model loaded successfully!


In [157]:
model.load_weights('action.h5')

# 10. Evaluation using Confusion Matrix and Accuracy

In [143]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [144]:
yhat = model.predict(X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 565ms/step


In [145]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

# Re-predict cleanly
yhat_probs = model.predict(X_test, verbose=0)
ytrue = np.argmax(y_test, axis=1).tolist()
yhat  = np.argmax(yhat_probs, axis=1).tolist()

print(f'✅ Test Accuracy: {accuracy_score(ytrue, yhat)*100:.2f}%')
print(f'Confusion Matrix:')
print(multilabel_confusion_matrix(ytrue, yhat))

✅ Test Accuracy: 65.22%
Confusion Matrix:
[[[16  0]
  [ 0  7]]

 [[ 7  8]
  [ 0  8]]

 [[15  0]
  [ 8  0]]]


In [146]:
multilabel_confusion_matrix(ytrue, yhat)

array([[[16,  0],
        [ 0,  7]],

       [[ 7,  8],
        [ 0,  8]],

       [[15,  0],
        [ 8,  0]]], dtype=int64)

In [147]:
accuracy_score(ytrue, yhat)

0.6521739130434783

# 11. Test in Real Time

In [52]:
# DATA_PATH = 'MP_Data'
# no_sequences = 50
# sequence_length = 30
# actions = np.array(['good','morning', 'hello','we','sign','human','rights', 'thanks'])
# import random
# random.seed(42)
# colors = [
  #  (random.randint(50,230), random.randint(50,230), random.randint(50,230))
   # for _ in actions
# print(f"✅ Actions: {list(actions)}")
# print(f"✅ Colors: {len(colors)}")

✅ Actions: ['good', 'morning', 'hello', 'we', 'sign', 'human', 'rights', 'thanks']
✅ Colors: 8


In [158]:
from scipy import stats
import random
print('✅ imports done')

✅ imports done


In [27]:
actions = get_recorded_actions(DATA_PATH)
print(f'✅ Signs loaded: {list(actions)}')

random.seed(42)
colors = [
    (random.randint(50,230), random.randint(50,230), random.randint(50,230))
    for _ in actions
]

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0, 60+num*45), (200, 90+num*45), (50,50,50), -1)
        cv2.rectangle(output_frame, (0, 60+num*45), (int(prob*200), 90+num*45), colors[num], -1)
        cv2.putText(output_frame, f'{actions[num]} {prob*100:.0f}%',
                    (5, 83+num*45), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1, cv2.LINE_AA)
    return output_frame

print('✅ prob_viz ready')

✅ Signs loaded: ['good', 'hello', 'human', 'morning', 'rights', 'sign', 'thanks', 'we']
✅ prob_viz ready


In [28]:
import subprocess
subprocess.run(['pip', 'install', 'pyttsx3'], 
               capture_output=True)
print("✅ installed!")

✅ installed!


In [29]:
import threading
import ctypes

def speak(word):
    def run():
        try:
            ctypes.windll.winmm.mciSendStringW(
                f'speak "{word}"', None, 0, None)
        except:
            import subprocess
            subprocess.Popen(
                ['powershell', '-Command',
                 f'Add-Type -AssemblyName System.Speech;'
                 f'$s=New-Object System.Speech.Synthesis.SpeechSynthesizer;'
                 f'$s.Rate=5;$s.Speak("{word}")'],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
    threading.Thread(target=run, daemon=True).start()

speak("hello")
print("✅ Did you hear it faster?")

✅ Did you hear it faster?


In [2]:
 #import tensorflow as tf
# tf.config.threading.set_intra_op_parallelism_threads(2)
#tf.config.threading.set_inter_op_parallelism_threads(2)
#print("✅ Done!")

In [74]:
import threading

import subprocess

import collections

from tensorflow.keras.models import load_model



# ✅ LOAD THE TRAINED MODEL

model = load_model('action.h5')

print('✅ Model loaded from action.h5')



def speak(word):


✅ Actions: ['good' 'hello' 'thanks']
✅ Label map: {'good': 0, 'hello': 1, 'thanks': 2}


In [77]:
# STEP 1 - Fix actions order
actions = np.array(['good', 'hello', 'thanks'])
print('✅ Fixed actions:', actions)

# STEP 2 - Verify
label_map = {'good': 0, 'hello': 1, 'thanks': 2}
print('✅ Label map:', label_map)

✅ Fixed actions: ['good' 'hello' 'thanks']
✅ Label map: {'good': 0, 'hello': 1, 'thanks': 2}


In [ ]:
import threading
import subprocess
import collections
from tensorflow.keras.models import load_model

# ✅ LOAD THE TRAINED MODEL
model = load_model('action.h5')
print('✅ Model loaded from action.h5')

def speak(word):
    def run():
        subprocess.Popen(
            ['powershell', '-Command',
             f'Add-Type -AssemblyName System.Speech;'
             f'$s=New-Object System.Speech.Synthesis.SpeechSynthesizer;'
             f'$s.Rate=5;$s.Speak("{word}")'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
    threading.Thread(target=run, daemon=True).start()

def clean_sentence(sentence):
    if not sentence: return sentence
    cleaned = [sentence[0]]
    for word in sentence[1:]:
        if word != cleaned[-1]: cleaned.append(word)
    return cleaned

# ✅ Faster settings
actions = get_recorded_actions(DATA_PATH)

THRESHOLDS = {
    'good': 0.88,
    'hello': 0.78,
    'thanks': 0.88
}
MIN_AGREE        = 18
COOLDOWN         = 35
cooldown_counter = 0
sequence         = []
sentence         = []
predictions      = collections.deque(maxlen=MIN_AGREE)
probabilities    = collections.deque(maxlen=MIN_AGREE)
last_confirmed_action = ""
last_spoken      = ""
frame_count      = 0

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)   # ✅ smaller = faster
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)  # ✅ smaller = faster
cap.set(cv2.CAP_PROP_FPS, 30)

cv2.namedWindow('OpenCV Feed', cv2.WINDOW_NORMAL)
cv2.moveWindow('OpenCV Feed', 0, 0)  # moves window to top-left corner

with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        model_complexity=0) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        frame_count += 1

        # ✅ Skip every other frame — faster and smoother
        if frame_count % 2 != 0:
            continue

        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        hand_visible = (results.left_hand_landmarks is not None or
                        results.right_hand_landmarks is not None)

        if not hand_visible:
            sequence = []
            predictions.clear()
            probabilities.clear()
            cv2.rectangle(image, (0,0), (320,40), (0,0,0), -1)
            cv2.putText(image, 'Show hand...', (5,28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (100,100,100), 2)
        else:
            keypoints = extract_keypoints(results)
            sequence.append(keypoints)
            sequence = sequence[-30:]

            if cooldown_counter > 0:
                cooldown_counter -= 1
            elif len(sequence) == 30:
                res = model.predict(
                    np.expand_dims(sequence, axis=0), verbose=0)[0]

                predicted_idx    = np.argmax(res)
                predicted_action = actions[predicted_idx]
                confidence       = float(res[predicted_idx])
                threshold        = THRESHOLDS.get(predicted_action, 0.90)

                predictions.append(predicted_idx)
                probabilities.append(res)
                image = prob_viz(res, actions, image, colors)

                if len(predictions) == MIN_AGREE:
                    if len(set(predictions)) != 1:
                        predictions.clear()
                        probabilities.clear()
                    else:
                        avg_confidence = float(np.mean([p[predicted_idx] for p in probabilities]))
                        avg_gap = float(np.mean([sorted(p, reverse=True)[0] - sorted(p, reverse=True)[1]
                                                  for p in probabilities]))

                        if (avg_confidence > threshold and confidence > threshold and
                                avg_gap > 0.23 and predicted_action != last_confirmed_action):
                            sentence.append(predicted_action)
                            speak(predicted_action)
                            last_spoken = predicted_action
                            last_confirmed_action = predicted_action
                            print(f'✅ {predicted_action} ({confidence*100:.1f}%)')
                            sequence = []
                            predictions.clear()
                            probabilities.clear()
                            cooldown_counter = COOLDOWN
                        else:
                            predictions.clear()
                            probabilities.clear()

            sentence = clean_sentence(sentence)
            if len(sentence) > 3:
                sentence = sentence[-3:]

            cv2.rectangle(image, (0,0), (320,40), (0,0,0), -1)
            cv2.putText(image,
                        ' '.join(sentence) if sentence else 'Waiting...',
                        (5,28), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (255,255,255), 2)

        buf_len = len(sequence)
        cv2.rectangle(image, (0,220), (int(buf_len/30*320), 240),
                      (0,255,100), -1)
        cv2.putText(image, f'Buf:{buf_len}/30  Said:{last_spoken}',
                    (5,235), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,0,0), 1)

        cv2.imshow('OpenCV Feed', image)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('c'):
            sentence = []
            predictions.clear()
            probabilities.clear()
            sequence = []
            cooldown_counter = 0
            last_confirmed_action = ""
            last_spoken = ""

cap.release()
cv2.destroyAllWindows()

In [73]:
# Check class distribution in training data
from collections import Counter

label_counts = Counter(labels)
for action, count in label_counts.items():
    print(f'{actions[action]}: {count} samples')

good: 50 samples
hello: 50 samples
thanks: 50 samples


In [31]:
speak("hello")
print("Did you hear hello?")

Did you hear hello?
